<a href="https://colab.research.google.com/github/Shineii86/MoeStickerBot/blob/main/notebooks/MoeStickerBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">
  <img src="https://capsule-render.vercel.app/api?type=waving&height=300&color=gradient&text=𝗠𝗼𝗲%20𝗦𝘁𝗶𝗰𝗸𝗲𝗿%20𝗕𝗼𝘁&fontAlignY=30&fontSize=100&desc=𝖢𝗈𝗅𝖺𝖻%20𝖤𝖽𝗂𝗍𝗂𝗈𝗇%20—%20𝖲𝖾𝗅𝖿‑𝖧𝗈𝗌𝗍%20𝖸𝗈𝗎𝗋%20𝖳𝖾𝗅𝖾𝗀𝗋𝖺𝗆%20𝖲𝗍𝗂𝖼𝗄𝖾𝗋%20𝖡𝗈𝗍&descSize=30" alt="Moe Sticker Bot">
  <p><b>Import LINE/Kakao · Create · Manage — all in one notebook</b></p>
</div>

---

## 🎯 Features

| Feature | Description |
|---------|-------------|
| 📥 **Import** | Import LINE & KakaoTalk sticker packs (including animated!) into Telegram |
| 🎨 **Create** | Create your own sticker sets from any image or video |
| 🛠️ **Manage** | Edit, reorder, add/remove stickers from your sets |
| 💾 **Download** | Download any Telegram sticker or GIF |
| 🔍 **Search** | Search previously imported sticker packs |

---

## 🤖 Bot Commands

| Command | What It Does |
|---------|-------------|
| `/start` | Welcome message & instructions |
| `/import` | Import a LINE or Kakao sticker pack |
| `/create` | Create a new sticker set from your images/videos |
| `/manage` | Edit your existing sticker sets |
| `/download` | Download Telegram stickers or GIFs |
| `/search` | Search imported sticker packs by keyword |
| `/help` | Show all available commands |

**💡 Tip:** Just send a LINE/Kakao link directly — the bot auto-detects it!

**Example links:**
```
https://store.line.me/stickershop/product/7673/ja
https://e.kakao.com/t/pretty-all-friends
https://emoticon.kakao.com/items/lV6K2fWmU7CpXlHcP9-ysQJx9rg=?referer=share_link
```

---

In [ ]:
#@title 🚀 Moe Sticker Bot — Configuration & Launch
# ╔══════════════════════════════════════════════════════════════════════╗
# ║          🚀 MOE STICKER BOT — Configuration & Launch               ║
# ║          ALL-IN-ONE CELL  ·  No UI boxes  ·  Full ANSI             ║
# ╚══════════════════════════════════════════════════════════════════════╝

# ==================== CONFIGURATION (EDIT HERE) ====================
ENABLE_DB       = True            # TiDB Cloud shared database (sticker data persists for everyone)
ENABLE_WEBAPP   = False           # Set True to expose WebApp via ngrok
WEBAPP_PORT     = 8080
NGROK_AUTHTOKEN = ""              # Required if ENABLE_WEBAPP=True
DATA_DIR        = "moe_sticker_bot_data"
LOG_LEVEL       = "info"          # debug, info, warn, error
HTTP_PROXY      = ""              # Optional http://proxy:port
AUTO_RESTART    = True            # Auto-restart bot on crash
MAX_RESTARTS    = 5               # Max restart attempts before giving up
KEEP_ALIVE      = True            # Print heartbeat every 10 min (fights Colab timeout)
# ===================================================================

# ========== ENCRYPTED BOT TOKEN (DO NOT EDIT) ==========
import base64 as _b64
_ENC_TOKEN = "dVhTY01aV15VREkDLjIAZQJsBAEtFjBREVwzLTY6FVlICGFTBB4dHwcTICAoJw=="
_a=bytes().fromhex("4d6f65537469");_b=bytes().fromhex("636b657273426f7432303236");_k=(_a+_b).decode()
BOT_TOKEN="".join(chr(ord(c)^ord(_k[i%len(_k)]))for i,c in enumerate(_b64.b64decode(_ENC_TOKEN).decode("latin-1")));del _a,_b,_k
# ========================================================

# ┌──────────────────────────────────────────────────────────────────┐
# │  📦 IMPORTS & DEPENDENCIES                                       │
# │  Standard library, third-party modules, and progress bars        │
# └──────────────────────────────────────────────────────────────────┘
import sys, time, subprocess, os, urllib.request, json, threading, signal, atexit
from itertools import cycle
import re
import requests
from tqdm.notebook import tqdm

# ┌──────────────────────────────────────────────────────────────────┐
# │  🎨 THEME-ADAPTIVE UI ENGINE                                    │
# │  Auto-detects light/dadark · Rich output · Debug mode            │
# └──────────────────────────────────────────────────────────────────┘
import sys as _sys, select as _sel, tty as _tty, termios as _term
from datetime import datetime as _dt

def _detect_theme():
    """Detect terminal background via OSC 11 query. Returns True if dark."""
    try:
        if not hasattr(_sys.stdin, 'fileno') or not _sys.stdin.isatty():
            raise Exception("not a tty")
        fd = _sys.stdin.fileno()
        old = _term.tcgetattr(fd)
        try:
            _tty.setraw(fd)
            _sys.stdout.write("\033]11;?\033\\")
            _sys.stdout.flush()
            resp = ""
            while True:
                r, _, _ = _sel.select([fd], [], [], 0.3)
                if not r: break
                ch = _sys.stdin.read(1)
                if not ch: break
                resp += ch
                if resp.endswith("\\") or len(resp) > 60: break
            if "rgb:" in resp:
                parts = resp.split("rgb:")[1].split("\\")[0].split("/")
                vals = [int(v[:2], 16) / 255.0 for v in parts[:3] if len(v) >= 2]
                if vals:
                    lum = 0.299 * vals[0] + 0.587 * vals[1] + 0.114 * vals[2]
                    return lum < 0.5
        finally:
            _term.tcsetattr(fd, _term.TCSADRAIN, old)
    except Exception:
        pass
    return True  # default to dark (Colab dark mode)

_DARK = _detect_theme()

class C:
    R   = '\033[0m';  B  = '\033[1m';  D  = '\033[2m';  BD  = '\033[22m'
    IT  = '\033[3m';  UL = '\033[4m';  BLINK = '\033[5m'
    # Text: bright colors for dark mode, standard for light
    GN  = '\033[92m' if _DARK else '\033[32m'
    CY  = '\033[96m' if _DARK else '\033[36m'
    RD  = '\033[91m' if _DARK else '\033[31m'
    YL  = '\033[93m' if _DARK else '\033[33m'
    MG  = '\033[95m' if _DARK else '\033[35m'
    BL  = '\033[94m' if _DARK else '\033[34m'
    WH  = '\033[97m' if _DARK else '\033[30m'
    BLK = '\033[30m' if _DARK else '\033[97m'
    # Named bright variants
    BGN = '\033[92m' if _DARK else '\033[32m'
    BCY = '\033[96m' if _DARK else '\033[36m'
    BRD = '\033[91m' if _DARK else '\033[31m'
    BYL = '\033[93m' if _DARK else '\033[33m'
    BMG = '\033[95m' if _DARK else '\033[35m'
    BBL = '\033[94m' if _DARK else '\033[34m'
    BWH = '\033[97m' if _DARK else '\033[30m'
    # Background strips
    BGRD = '\033[41m'; BGGN = '\033[42m'; BGYL = '\033[43m'; BGBL = '\033[44m'
    BGMG = '\033[45m'; BGCY = '\033[46m'

# Cleanup detection helpers
del _sys, _sel, _tty, _term


# ─── Timestamp Helper ────────────────────────────────────────
def _ts():
    return f"{C.D}{_dt.now().strftime('%H:%M:%S')}{C.R}"

# ─── Output Helpers ──────────────────────────────────────────
def success(m):
    print(f"  {C.B}{C.BGGN}{C.WH} ✓ {C.R} {C.B}{C.GN}{m}{C.R}  {_ts()}")

def error(m):
    print(f"  {C.B}{C.BGRD}{C.WH} ✗ {C.R} {C.B}{C.RD}{m}{C.R}  {_ts()}")

def info(m):
    print(f"  {C.B}{C.BGBL}{C.WH} ℹ {C.R} {C.CY}{m}{C.R}  {_ts()}")

def warn(m):
    print(f"  {C.B}{C.BGYL}{C.WH} ⚠ {C.R} {C.B}{C.YL}{m}{C.R}  {_ts()}")

def debug(m):
    if LOG_LEVEL == "debug":
        print(f"  {C.D}  ◆ {m}{C.R}")


def header(t):
    w = 56
    print(f"\n  {C.B}{C.BCY}{'─'*w}{C.R}")
    print(f"  {C.B}{C.BCY}  {t}{C.R}")
    print(f"  {C.B}{C.BCY}{'─'*w}{C.R}\n")

def spinner(msg, dur=2):
    frames = ['⠋','⠙','⠹','⠸','⠼','⠴','⠦','⠧','⠇','⠏']
    end = time.time() + dur
    i = 0
    while time.time() < end:
        f = frames[i % len(frames)]
        sys.stdout.write(f'\r  {C.BCY}{f}{C.R} {C.B}{msg}{C.R}  ')
        sys.stdout.flush()
        time.sleep(0.08)
        i += 1
    sys.stdout.write(f'\r  {C.B}{C.GN}✔{C.R} {C.B}{msg}{C.R}        \n')

def progress_bar(items, prefix="", width=30):
    """Simple progress bar for iterations."""
    total = len(items)
    for i, item in enumerate(items):
        pct = (i + 1) / total
        filled = int(width * pct)
        bar = f"{C.GN}{'█'*filled}{C.D}{'░'*(width-filled)}{C.R}"
        sys.stdout.write(f'\r  {prefix} {bar} {C.B}{pct*100:.0f}%{C.R}')
        sys.stdout.flush()
        yield item
    sys.stdout.write(f'\r  {prefix} {C.GN}{"█"*width}{C.R} {C.B}100%{C.R}\n')

def box(title, lines, color=BCY):
    """Print a styled box with title and content lines."""
    w = 56
    print(f"\n  {C.B}{color}╭{'─'*(w-2)}╮{C.R}")
    print(f"  {C.B}{color}│{C.R} {C.B}{title:<{w-4}}{C.R} {C.B}{color}│{C.R}")
    print(f"  {C.B}{color}├{'─'*(w-2)}┤{C.R}")
    for line in lines:
        print(f"  {C.B}{color}│{C.R}  {line:<{w-5}}{C.B}{color}│{C.R}")
    print(f"  {C.B}{color}╰{'─'*(w-2)}╯{C.R}\n")

def kv(key, val, indent=4):
    """Print a key-value pair with styling."""
    print(f"{' '*indent}{C.GN}●{C.R} {C.B}{key}{C.R}: {val}")

def banner_kv(key, val):
    """Print a key-value inside banner."""
    return f"  {C.GN}●{C.R} {C.B}{key}{C.R}: {val}"

# ┌──────────────────────────────────────────────────────────────────┐
# │  🖥️  STARTUP BANNER                                              │
# │  Display welcome screen with bot status and configuration info   │
# └──────────────────────────────────────────────────────────────────┘
_theme_name = "DARK" if _DARK else "LIGHT"
print()
print(f"  {C.B}{C.BCY}╔══════════════════════════════════════════════════════════╗{C.R}")
print(f"  {C.B}{C.BCY}║{C.R}  {C.B}{C.BWH}🚀  Moe Sticker Bot{C.R}  {C.D}— Colab Edition{C.R}              {C.B}{C.BCY}║{C.R}")
print(f"  {C.B}{C.BCY}║{C.R}  {C.D}Import LINE/Kakao · Create · Manage · Download{C.R}       {C.B}{C.BCY}║{C.R}")
print(f"  {C.B}{C.BCY}╠══════════════════════════════════════════════════════════╣{C.R}")
print(f"  {C.B}{C.BCY}║{C.R}  {C.B}{C.BGGN}{C.WH} PRE-CONFIGURED {C.R}  Bot token · DB · Deps          {C.B}{C.BCY}║{C.R}")
print(f"  {C.B}{C.BCY}║{C.R}  {C.B}{C.BGRD}{C.WH} ⚠ WARNING {C.R}       Don't edit — just run!         {C.B}{C.BCY}║{C.R}")
print(f"  {C.B}{C.BCY}║{C.R}  {C.D}@MoeStickersBot → t.me/MoeStickersBot{C.R}                {C.B}{C.BCY}║{C.R}")
print(f"  {C.B}{C.BCY}╚══════════════════════════════════════════════════════════╝{C.R}")
print()
kv("Bot token",   f"{C.GN}encrypted & locked{C.R}")
kv("Database",    f"{C.GN}TiDB Cloud (shared){C.R}")
kv("Keep-alive",  f"{C.GN}heartbeat every 10 min{C.R}")
kv("Auto-restart",f"up to {C.B}{MAX_RESTARTS}{C.R} attempts on crash")
kv("Theme",       f"{C.BCY}{_theme_name}{C.R} mode")
kv("Debug",       f"{C.BYL}ON{C.R}" if LOG_LEVEL == "debug" else f"{C.D}off{C.R}")
print()
print(f"  {C.B}{C.GN}✨ Ready! Starting setup...{C.R}")
print()
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔧  PHASE 1 — SETUP ENVIRONMENT & BUILD BOT                   ║
# ║  Install deps · Connect DB · Download Go · Build binary         ║
# ╚══════════════════════════════════════════════════════════════════╝

# ┌── 📥 Install System Dependencies ────────────────────────────────┐
header("Installing System Dependencies")
spinner("Updating packages", 1)
!apt-get update -qq 2>/dev/null
!apt-get install -y -qq imagemagick libarchive-tools ffmpeg curl gifsicle python3 exiv2 2>/dev/null
success("Core packages installed")

# ┌── 🗄️  Connect to TiDB Cloud Database ───────────────────────────┐
if ENABLE_DB:
    header("Connecting to TiDB Cloud Database")
    !which mysql >/dev/null 2>&1 || apt-get install -y -qq mysql-client 2>/dev/null
    _tidb_host = "gateway01.ap-southeast-1.prod.aws.tidbcloud.com"
    _tidb_port = "4000"
    _tidb_user = "3yvemDZyfJVpUSg.root"
    _tidb_pass = "JnamvZsoCC7cOitX"
    _tidb_db   = "MoeStickersBot_db"
    _test = subprocess.run(
        ["mysql", "-h", _tidb_host, "-P", _tidb_port,
         "-u", _tidb_user, f"-p{_tidb_pass}", "--ssl-mode=REQUIRED",
         "-e", f"USE `{_tidb_db}`;"],
        capture_output=True, text=True)
    if _test.returncode == 0:
        success(f"TiDB Cloud connected — {C.B}{_tidb_db}{C.R} (shared)")
    else:
        warn(f"DB connection: {_test.stderr.strip()[:100]}")
else:
    info("Database disabled")

# ┌── 🐹 Download & Setup Go Language ──────────────────────────────┐
GO_VERSION = "go1.22.4"
GO_URL = f"https://go.dev/dl/{GO_VERSION}.linux-amd64.tar.gz"
if not os.path.exists("/usr/local/go/bin/go") or GO_VERSION not in subprocess.getoutput("go version"):
    print(f"{C.CY}⬇ Downloading {GO_VERSION}...{C.R}")
    with tqdm(unit='B', unit_scale=True, desc=f"{C.BCY}Go{C.R}") as t:
        urllib.request.urlretrieve(GO_URL, "go.tar.gz", reporthook=lambda b,bs,total: t.update(b*bs-t.n))
    !rm -rf /usr/local/go && tar -C /usr/local -xzf go.tar.gz
else:
    info(f"{GO_VERSION} already installed — skipping download")
os.environ['PATH'] += ":/usr/local/go/bin"
os.environ['GOPATH'] = "/root/go"
os.environ['GO111MODULE'] = "on"
os.environ['GOFLAGS'] = "-mod=mod"
!mkdir -p $GOPATH
success(f"Go {subprocess.getoutput('go version').split()[2]} ready")

# ┌── 🐍 Install Python Helper Scripts ─────────────────────────────┐
header("Installing Python Helpers")
helpers = [("msb_emoji.py","Emoji"), ("msb_kakao_decrypt.py","Kakao"), ("msb_rlottie.py","Lottie")]
for f,desc in helpers:
    !wget -q https://raw.githubusercontent.com/Shineii86/MoeStickersBot/master/tools/{f} -O /usr/local/bin/{f}
    !chmod +x /usr/local/bin/{f}
    print(f"  {C.GN}✓{C.R} {desc}")
success("Helpers installed")

# ┌── 🔨 Build MoeStickersBot Binary ───────────────────────────────┐
header("Building MoeStickersBot")
!rm -rf MoeStickersBot
!git clone --depth 1 https://github.com/Shineii86/MoeStickersBot.git 2>&1 | grep -v "Cloning"
%cd MoeStickersBot
spinner("Patching TiDB TLS support", 1)
!sed -i 's/params\["autocommit"\] = "1"/params["autocommit"] = "1"\n\tparams["tls"] = "true"/' core/database.go
success("TLS patch applied to database.go")
spinner("Downloading Go modules", 2)
!go mod download
spinner("Compiling binary", 3)
!go build -ldflags="-s -w" -o MoeStickersBot cmd/MoeStickersBot/main.go
if os.path.exists("MoeStickersBot"):
    sz = os.path.getsize("MoeStickersBot")/1024/1024
    success(f"Build complete — Binary: {sz:.1f} MB")
else:
    error("Build failed — check git clone / go build output above")
    sys.exit(1)

# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  PHASE 2 — CONFIGURE & PREPARE LAUNCH                      ║
# ║  Validate token · ngrok tunnel · Build CLI · Log colorizer      ║
# ╚══════════════════════════════════════════════════════════════════╝

if not BOT_TOKEN:
    error("BOT_TOKEN is not set! Edit the variable at the top of this cell and re-run.")
    sys.exit(1)

if not re.match(r'^\d+:[A-Za-z0-9_-]{35,}$', BOT_TOKEN):
    warn("BOT_TOKEN format looks wrong — expected  123456:ABC…  (digits:35+chars)")

header("Configuration")
print(f"  {C.GN}✓{C.R} BOT_TOKEN    = {BOT_TOKEN[:8]}...{BOT_TOKEN[-4:]}")
print(f"  {C.GN}✓{C.R} LOG_LEVEL    = {LOG_LEVEL}")
print(f"  {C.GN}✓{C.R} DATA_DIR     = {DATA_DIR}")
print(f"  {C.GN}✓{C.R} AUTO_RESTART = {AUTO_RESTART}  (max {MAX_RESTARTS}x)")
print(f"  {C.GN}✓{C.R} KEEP_ALIVE   = {KEEP_ALIVE}")

if ENABLE_WEBAPP and not NGROK_AUTHTOKEN:
    warn("WebApp enabled but no ngrok token — disabled")
    ENABLE_WEBAPP = False

# ┌── 🌐 Setup ngrok Tunnel (if WebApp enabled) ────────────────────┐
WEBAPP_URL = ""
ngrok_proc = None
if ENABLE_WEBAPP:
    header("Setting up ngrok Tunnel")
    if not os.path.exists("./ngrok"):
        !wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz && tar -xzf ngrok*.tgz && chmod +x ngrok
    !./ngrok config add-authtoken {NGROK_AUTHTOKEN}
    !pkill -f ngrok || true
    ngrok_proc = subprocess.Popen(["./ngrok", "http", str(WEBAPP_PORT), "--log", "stdout"],
                                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    spinner("Starting ngrok", 3)
    for _ in range(15):
        try:
            r = requests.get("http://127.0.0.1:4040/api/tunnels", timeout=2)
            if r.status_code == 200:
                tuns = r.json()['tunnels']
                if tuns:
                    WEBAPP_URL = tuns[0]['public_url']
                    success(f"ngrok URL: {WEBAPP_URL}")
                    break
        except Exception: pass
        time.sleep(1)
    else:
        error("Could not retrieve ngrok URL — WebApp disabled")
        ENABLE_WEBAPP = False

# ┌── 🧩 Build Bot Command Line Arguments ──────────────────────────┐
def build_cmd():
    cmd = ["./MoeStickersBot",
           f"--bot_token={BOT_TOKEN}",
           f"--log_level={LOG_LEVEL}",
           f"--data_dir={DATA_DIR}"]
    if ENABLE_DB:
        cmd.extend(["--db_addr=gateway01.ap-southeast-1.prod.aws.tidbcloud.com:4000", "--db_user=3yvemDZyfJVpUSg.root", "--db_pass=JnamvZsoCC7cOitX"])
    if ENABLE_WEBAPP and WEBAPP_URL:
        cmd += [f"--webapp_url={WEBAPP_URL}", f"--webapp_listen_addr=0.0.0.0:{WEBAPP_PORT}"]
    return cmd

if HTTP_PROXY:
    os.environ['HTTP_PROXY'] = HTTP_PROXY
    os.environ['HTTPS_PROXY'] = HTTP_PROXY

cmd_line = build_cmd()
print(f"{C.D}Command: {' '.join(cmd_line).replace(BOT_TOKEN, '[REDACTED]')}{C.R}")

# ┌── 🌈 Colorize Log Lines (ANSI formatting) ──────────────────────┐
def colorize(line):
    """Apply syntax highlighting to bot log output."""
    # Log level badges (match whole word to avoid partial replacements)
    line = re.sub(r'\bINFO\b',    f'{C.BGBL}{C.WH} INFO {C.R}', line)
    line = re.sub(r'\bWARNING\b', f'{C.BGYL}{C.BLK} WARN {C.R}', line)
    line = re.sub(r'\bERROR\b',   f'{C.BGRD}{C.WH} ERR  {C.R}', line)
    line = re.sub(r'\bDEBUG\b',   f'{C.D} DBG  {C.R}', line)
    line = re.sub(r'\bFATAL\b',   f'{C.B}{C.BGRD}{C.WH} FATAL {C.R}', line)
    # Bot status keywords
    line = line.replace('Bot OK',  f'{C.B}{C.GN}● Bot OK{C.R}')
    line = line.replace('Success', f'{C.B}{C.GN}✔ Success{C.R}')
    # Sticker operations
    line = line.replace('one sticker commited',      f'{C.GN}✔ committed{C.R}')
    line = line.replace('Failed to add one sticker', f'{C.BRD}{C.B}✘ failed{C.R}')
    line = line.replace('STICKER_VIDEO_LONG',  f'{C.BYL}{C.B}⚠ VIDEO_LONG{C.R}')
    line = line.replace('safe mode',           f'{C.MG}◈ safe mode{C.R}')
    line = line.replace('convertKakaoAnimated OK', f'{C.GN}✔ converted{C.R}')
    # Database operations
    line = line.replace('MariaDB OK',    f'{C.GN}✔ DB connected{C.R}')
    line = line.replace('Insert LineS',  f'{C.CY}→ Insert Line{C.R}')
    line = line.replace('Insert UserS',  f'{C.CY}→ Insert User{C.R}')
    # IP addresses and ports (dim them)
    line = re.sub(r'(\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}:\d+)', f'{C.D}\\1{C.R}', line)
    return line
# ┌── 🧹 Cleanup & Shutdown Handler ────────────────────────────────┐
_process_ref = [None]

def cleanup():
    p = _process_ref[0]
    if p and p.poll() is None:
        p.terminate()
        try: p.wait(timeout=5)
        except subprocess.TimeoutExpired: p.kill()
    if ngrok_proc and ngrok_proc.poll() is None:
        ngrok_proc.terminate()
    print(f"\n{C.BGYL}Shutting down...{C.R}")
    success("Bot stopped.")

atexit.register(cleanup)

# ┌── 💓 Keep-Alive Heartbeat Thread ───────────────────────────────┐
_ka_stop = threading.Event()
def _keep_alive():
    beat = 0
    while not _ka_stop.wait(600):
        beat += 1
        print(f"{C.D}[keep-alive #{beat} — {time.strftime('%H:%M:%S')}]{C.R}")
if KEEP_ALIVE:
    threading.Thread(target=_keep_alive, daemon=True).start()

# ╔══════════════════════════════════════════════════════════════════╗
# ║  🚀  PHASE 3 — LAUNCH BOT (with auto-restart)                  ║
# ║  Start process · Stream logs · Auto-restart on crash            ║
# ╚══════════════════════════════════════════════════════════════════╝
header("Launching Bot")

restart_count = 0
while True:
    _stderr_log = os.path.join(DATA_DIR, "stderr.log") if os.path.isdir(DATA_DIR) else "/tmp/msb_stderr.log"
    _stderr_f = open(_stderr_log, "w")
    process = subprocess.Popen(
        build_cmd(),
        stdout=subprocess.PIPE, stderr=_stderr_f,
        cwd=os.getcwd(), bufsize=1, universal_newlines=True
    )
    _process_ref[0] = process

    spinner("Starting bot", 2)
    time.sleep(2)

    if process.poll() is not None:
        rc = process.returncode
        _stderr_f.close()
        error(f"Bot exited immediately (code {rc})")
        try:
            with open(_stderr_log) as f:
                err = f.read().strip()
            if err:
                print(f"\n{C.B}{C.BRD}── STDERR ──{C.R}")
                for eline in err.split('\n')[-20:]:
                    print(f"  {C.BRD}{eline}{C.R}")
                print(f"{C.B}{C.BRD}────────────{C.R}\n")
            else:
                warn("No stderr output captured")
        except Exception:
            warn("Could not read stderr log")
        sys.exit(1)

    success(f"Bot is RUNNING — PID {process.pid}"
            + (f"  [restart #{restart_count}]" if restart_count else ""))
    print(f"{C.B}{C.BGN}📱 Send /start to your bot on Telegram!{C.R}")
    if WEBAPP_URL:
        print(f"{C.B}{C.BCY}🌐 WebApp: {WEBAPP_URL}{C.R}")
    print()

    header("Live Bot Logs — Press ■ (Stop) to terminate")
    print(f"  {C.D}Logs appear below in real-time.{C.R}\n")
    try:
        for line in process.stdout:
            line = line.rstrip()
            if line:
                print(colorize(line))
    except KeyboardInterrupt:
        warn("Interrupted by user")
        break

    rc = process.wait()
    _stderr_f.close()
    # Show stderr on unexpected exit
    try:
        with open(_stderr_log) as f:
            err = f.read().strip()
        if err and rc != 0:
            print(f"\n{C.BRD}── Bot STDERR (last 10 lines) ──{C.R}")
            for eline in err.split('\n')[-10:]:
                print(f"  {C.BRD}{eline}{C.R}")
            print(f"{C.BRD}──────────────────────────────{C.R}\n")
    except Exception:
        pass

    if not AUTO_RESTART or restart_count >= MAX_RESTARTS:
        warn(f"Bot exited (code {rc}). AUTO_RESTART={AUTO_RESTART}, restarts={restart_count}/{MAX_RESTARTS}.")
        break

    restart_count += 1
    warn(f"Bot exited (code {rc}) — restarting in 5s… (attempt {restart_count}/{MAX_RESTARTS})")
    time.sleep(5)
_ka_stop.set()
cleanup()

---

## 🔧 Troubleshooting

| Problem | Solution |
|---------|----------|
| **"Bot exited immediately"** | Check if your `BOT_TOKEN` is correct |
| **"Database not enabled"** | Normal! Bot works fine without database |
| **Bot stops after ~90 min** | Free Colab disconnects — `KEEP_ALIVE=True` helps; use Colab Pro for longer sessions |
| **Sticker import fails** | Try again — Telegram rate-limits sometimes. Bot auto-retries |
| **"STICKER_VIDEO_LONG"** | Bot handles this automatically via safe mode |
| **WebApp not working** | Set `ENABLE_WEBAPP = True` + valid `NGROK_AUTHTOKEN` |
| **"go build" fails** | Run cell again — Go module download may have timed out |
| **Bot keeps restarting** | Set `AUTO_RESTART = False` to debug; check logs above |

---

## 💡 Pro Tips

1. **Keep Colab Alive** — `KEEP_ALIVE = True` prints a heartbeat every 10 min; keep tab open and interact every 30-45 min
2. **Auto-Restart** — `AUTO_RESTART = True` + `MAX_RESTARTS = 5` recovers from crashes automatically
3. **Animated Kakao** — Use share links (KakaoTalk → Share → Copy Link) for animation support
4. **Mixed Sets** — Put animated + static stickers in the same Telegram sticker set
5. **Re-run to Update** — Run the cell again to pull latest bot code and rebuild
6. **Debug Mode** — Set `LOG_LEVEL = "debug"` for detailed logs
7. **Faster Builds** — Binary is now stripped (`-ldflags="-s -w"`) — smaller & faster to start

---

## ❓ FAQ

**Q: Is this free?** → Yes! Colab is free, bot is open-source.

**Q: 24/7 hosting?** → Free Colab disconnects after ~90 min. Use a VPS for 24/7.

**Q: Need WebApp?** → No, optional. Bot works fully without it.

**Q: Stickers saved forever?** → Yes! Once in Telegram, they stay even if bot goes offline.

**Q: What changed in this version?**
- Go upgraded to **1.22.4** (auto-skips re-download if already installed)
- **Auto-restart** on crash with configurable max attempts
- **Keep-alive** heartbeat thread to fight Colab timeouts
- BOT_TOKEN **format validation** before launch
- `HTTP_PROXY` now sets both `HTTP_PROXY` and `HTTPS_PROXY`
- Binary built with `-ldflags="-s -w"` — stripped & smaller
- ngrok tunnel wait extended to 15 retries with timeout
- `atexit` cleanup — bot stops cleanly even on unexpected exits

---

<div align="center">
  <img src="https://capsule-render.vercel.app/api?type=waving&color=gradient&customColorList=12,14,20,24,27&height=100&section=footer" width="100%">
  <p>Made with ❤️ for the sticker community</p>
</div>


---

## 🔐 Owner-Only: Update Bot Token

> ⚠️ **STOP!** This section is for the **bot owner only**.
> 
> **Regular users:** Skip this. Your bot is already configured.
> 
> **If you're not @Shineii86 — DO NOT touch this section.**

If you need to change the bot token, run the cell below with your new token.

In [ ]:
#@title 🔐 Owner-Only: Encrypt / Decrypt Bot Token
#@markdown ---
#@markdown ### 🔑 Owner Verification
#@markdown Enter your secret owner key to unlock this tool.
OWNER_KEY = ""  #@param {type:"string"}

#@markdown ---
#@markdown ### ⚙️ Action
ACTION = "encrypt"  #@param ["encrypt", "decrypt"]
#@markdown *`encrypt` = plain token → encrypted string · `decrypt` = encrypted string → plain token*

#@markdown ---
#@markdown ### 🔓 Encrypt — Paste your raw bot token here
#@markdown *Get one from [@BotFather](https://t.me/BotFather) → `/newbot`*
PLAIN_TOKEN = ""  #@param {type:"string"}

#@markdown ---
#@markdown ### 🔒 Decrypt — Paste your encrypted token here
ENC_TOKEN = ""  #@param {type:"string"}

#@markdown ---

# ┌──────────────────────────────────────────────────────────────────┐
# │  📦 IMPORTS                                                      │
# │  Cryptographic hashing, encoding, and IPython display utils      │
# └──────────────────────────────────────────────────────────────────┘
import base64, hashlib, re, sys
from IPython.display import display, HTML

_OWNER_HASH = "4781a06dee7845726be8a8180e22df8984680de58704a4880b3ae996c97233ef"

# ┌──────────────────────────────────────────────────────────────────┐
# │  🔐 ENCRYPTION / DECRYPTION CORE FUNCTIONS                       │
# │  XOR cipher with rotating key, token format validation           │
# └──────────────────────────────────────────────────────────────────┘

def _xor(text, k):
    return "".join(chr(ord(c) ^ ord(k[i % len(k)])) for i, c in enumerate(text))

def _cipher_key():
    _a = bytes().fromhex("4d6f65537469")
    _b = bytes().fromhex("636b657273426f7432303236")
    return (_a + _b).decode()

def encrypt_token(raw):
    k = _cipher_key()
    return base64.b64encode(_xor(raw, k).encode("latin-1")).decode()

def decrypt_token(enc):
    k = _cipher_key()
    return _xor(base64.b64decode(enc).decode("latin-1"), k)

# ┌──────────────────────────────────────────────────────────────────┐
# │  ▶️  MAIN EXECUTION — Verify Key & Process Token                 │
# │  Owner verification → encrypt or decrypt based on ACTION         │
# └──────────────────────────────────────────────────────────────────┘

print("═" * 50)
print("  🔐 Owner-Only Token Tool")
print("═" * 50)

if not OWNER_KEY:
    print("❌ Enter your Owner Key above and run again.")
elif hashlib.sha256(OWNER_KEY.encode()).hexdigest() != _OWNER_HASH:
    print("🔒 Access denied — incorrect key.")
else:
    print("  ✅ Key verified!")
    print()

    if ACTION == "encrypt":
        if not PLAIN_TOKEN:
            print("❌ PLAIN_TOKEN is empty. Paste your raw bot token above.")
        else:
            result = encrypt_token(PLAIN_TOKEN)
            print("═" * 50)
            print("  ✅ Encrypted Token — copy the line below")
            print("═" * 50)
            print(f"  {result}")
            print()
            print("📋 In the main cell, find:")
            print('     _ENC_TOKEN = "..."'
)
            print(f'   Replace with:\n     _ENC_TOKEN = "{result}"')
            print("═" * 50)

    elif ACTION == "decrypt":
        if not ENC_TOKEN:
            print("❌ ENC_TOKEN is empty. Paste your encrypted string above.")
        else:
            try:
                result = decrypt_token(ENC_TOKEN)
                valid  = bool(re.match(r"^\d+:[A-Za-z0-9_-]{35,}$", result))
                status = "✅ Valid Telegram token format" if valid else "⚠️  Format looks unusual — double-check"
                print("═" * 50)
                print("  🔓 Decrypted Token")
                print("═" * 50)
                print(f"  {result}")
                print()
                print(f"  {status}")
                print("═" * 50)
            except Exception as e:
                print(f"❌ Decryption failed: {e}")
